# 6 Searching architectures, twice

`1_pytorch_intro/04_search` ran one ray.tune search and read it: which axis actually
moved the metric, on a frozen encoder's cache. This notebook runs the same tool, the
same way, on the composition family `05_gates_highways_moe` measured on ETTh1's plain
window and lost with -- **on `pedestrian_counts` instead**, which has a daily cycle for
a router to find. D17 (main_principles.md): the point is not a different result for its
own sake, it is exercising the search habit a second time, in a modality where nobody
handed you a frozen representation to search on top of.

Before writing this, the same check L1 did before committing to a search dataset
(main_principles.md §3.4b, where EuroSAT's encoder axis turned out too flat to search):
**does this search space actually spread here, or would it be flat like `05`'s ETTh1
window was?** Section 6.1 answers that with the search itself, not with an assumption.


In [1]:
import os
import warnings

# same fix 1_pytorch_intro/04_search needs: ray propagates `uv run` to its workers,
# which then expect pyproject.toml in the notebook's own directory. Must happen before
# `import ray`.
os.environ.setdefault("RAY_ENABLE_UV_RUN_RUNTIME_ENV", "0")
warnings.filterwarnings("ignore", message=".*pkg_resources.*")

import sys
import time
from pathlib import Path

import datasets
import numpy as np
import torch
from datasets import load_dataset
from loguru import logger
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from vectormesh.components import Gate, Highway, MoE, NeuralNet

from ett import window_index
from notebooktester import param

logger.remove()
logger.add(sys.stderr, level="WARNING")
datasets.disable_progress_bars()

W, H = 168, 24            # same window 03_decomposition_and_nbeats uses on pedestrian
EPOCHS = param(15, test=1)
N_TRIALS = param(24, test=2)
SEEDS = param(5, test=2)


## 6.1 The baseline, first

Same discipline every notebook in this course insists on: name the number a search has
to beat before running one.


In [2]:
def load_pedestrian_windows(w: int = W, h: int = H, n_series: int = 8, stride: int = 6) -> dict:
    # identical to 03_decomposition_and_nbeats's load_pedestrian + split_scale_window,
    # inlined here because ray workers each call this independently (see 6.2) and need
    # it self-contained rather than a notebook-global closure.
    ds = load_dataset(
        "Monash-University/monash_tsf", revision="refs/convert/parquet",
        data_files={"test": "pedestrian_counts/test/0000.parquet"},
    )["test"]
    series = []
    for row in ds:
        s = np.asarray(row["target"], dtype="float32")
        s = s[~np.isnan(s)]
        if len(s) >= 6 * (w + h):
            series.append(torch.tensor(s))
        if len(series) >= n_series:
            break

    parts = {"train": [[], []], "val": [[], []], "test": [[], []]}
    for s in series:
        n = len(s)
        cuts = {"train": (0, int(0.6 * n)), "val": (int(0.6 * n), int(0.8 * n)),
                "test": (int(0.8 * n), n)}
        mu = s[: cuts["train"][1]].mean()
        sigma = s[: cuts["train"][1]].std().clamp(min=1e-6)
        for split, (a, b) in cuts.items():
            v = (s[a:b] - mu) / sigma
            win = v[window_index(len(v), w + h, stride)]
            parts[split][0].append(win[:, :w])
            parts[split][1].append(win[:, w:])
    return {k: (torch.cat(v[0]), torch.cat(v[1])) for k, v in parts.items()}


pedestrian = load_pedestrian_windows()
xtr, ytr = pedestrian["train"]
xva, yva = pedestrian["val"]
print(f"train {tuple(xtr.shape)}   val {tuple(xva.shape)}")


def mae(y: torch.Tensor, yhat: torch.Tensor) -> float:
    return float((y - yhat).abs().mean())


def make_model(mechanism: str, hidden: int, n_experts: int = 4) -> nn.Module:
    # the exact family 05_gates_highways_moe built: Linear -> GELU -> [mechanism] -> Linear.
    # "plain" is the capacity-matched control from that notebook's 5.6, folded into the
    # search space here as just another mechanism choice instead of a separate control run.
    if mechanism == "plain":
        mid = nn.Identity()
    elif mechanism == "gate":
        mid = Gate(hidden_size=hidden)
    elif mechanism == "highway":
        mid = Highway(transform=NeuralNet(hidden, hidden), hidden_size=hidden)
    elif mechanism == "moe":
        mid = MoE(experts=[nn.Linear(hidden, hidden) for _ in range(n_experts)],
                  hidden_size=hidden, out_size=hidden)
    else:
        raise ValueError(mechanism)
    return nn.Sequential(nn.Linear(W, hidden), nn.GELU(), mid, nn.Linear(hidden, H))


def run(model: nn.Module, epochs: int = EPOCHS, lr: float = 1e-3, batch_size: int = 64) -> float:
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    loader = DataLoader(TensorDataset(xtr, ytr), batch_size=batch_size, shuffle=True)
    for _ in range(epochs):
        for xb, yb in loader:
            opt.zero_grad()
            loss_fn(model(xb), yb).backward()
            opt.step()
    model.eval()
    with torch.no_grad():
        return mae(yva, model(xva))


scores = []
start = time.time()
for seed in range(SEEDS):
    torch.manual_seed(seed)
    scores.append(run(make_model("plain", hidden=64)))

BASELINE = float(np.mean(scores))
BASELINE_STD = float(np.std(scores))
print(f"seeds: {[f'{s:.4f}' for s in scores]}")
print(f"\nBASELINE (plain, hidden=64)  {BASELINE:.4f} +/- {BASELINE_STD:.4f}   "
      f"({time.time() - start:.0f}s for {SEEDS} runs)")
print(f"anything inside +/-{2 * BASELINE_STD:.4f} of that is not a result.")


train (72165, 168)   val (23888, 168)


seeds: ['0.2844', '0.2791', '0.2788', '0.2790', '0.2812']

BASELINE (plain, hidden=64)  0.2805 +/- 0.0021   (18s for 5 runs)
anything inside +/-0.0043 of that is not a result.


## 6.2 The search space

`mechanism` is the axis this notebook exists to test: plain (no routing, the control),
gate, highway, moe. `hidden` and `lr` are the same two axes L1's search always includes,
because width and learning rate can confound a comparison between architectures if they
are not searched alongside it. `n_experts` only matters when `mechanism == "moe"` --
same non-conditional-hyperopt tradeoff L1 notebook 04 names explicitly for its own
`hidden_dim`/`n_hidden_layers` pair, stated here for the same reason: a true conditional
search space is available in ray but not worth the complexity for one extra axis.


In [3]:
from ray import tune
from ray.tune.search.hyperopt import HyperOptSearch


def trainable(config):
    # ray runs this in its own process, so everything it needs is loaded inside --
    # same reason 1_pytorch_intro/04_search's trainable calls VectorCache.from_hub
    # itself rather than closing over a notebook-level cache.
    data = load_pedestrian_windows(config["w"], config["h"])
    xtr, ytr = data["train"]
    xva, yva = data["val"]

    torch.manual_seed(0)
    model = make_model(config["mechanism"], config["hidden"], config.get("n_experts", 4))
    opt = torch.optim.Adam(model.parameters(), lr=config["lr"])
    loss_fn = nn.MSELoss()
    loader = DataLoader(TensorDataset(xtr, ytr), batch_size=64, shuffle=True)
    for _ in range(config["epochs"]):
        for xb, yb in loader:
            opt.zero_grad()
            loss_fn(model(xb), yb).backward()
            opt.step()

    model.eval()
    with torch.no_grad():
        val_mae = float((model(xva) - yva).abs().mean())
    tune.report({"MAE": val_mae})


param_space = {
    "mechanism": tune.choice(["plain", "gate", "highway", "moe"]),
    "hidden": tune.choice([32, 64, 128]),
    "n_experts": tune.choice([2, 4, 8]),   # ignored unless mechanism == "moe"
    "lr": tune.loguniform(1e-4, 3e-3),
    "w": W,
    "h": H,
    "epochs": EPOCHS,
}

tuner = tune.Tuner(
    trainable,
    param_space=param_space,
    tune_config=tune.TuneConfig(
        metric="MAE",
        mode="min",
        search_alg=HyperOptSearch(),
        num_samples=N_TRIALS,
    ),
    run_config=tune.RunConfig(
        storage_path=Path("ray_results").resolve(),
        name="pedestrian_gate_search",
        verbose=1,   # same reasoning as 1_pytorch_intro/04_search 4.1: this, not an
                     # explicit reporter, is what makes the live status table appear
    ),
)

start = time.time()
results = tuner.fit()
print(f"{N_TRIALS} trials in {time.time() - start:.0f}s")


2026-08-05 14:43:18,131	INFO tune.py:1001 -- Wrote the latest version of all result files and experiment state to '/Users/rgrouls/code/courses/MADS-MachineLearning-course/notebooks/2_timeseries/ray_results/pedestrian_gate_search' in 0.0076s.


2026-08-05 14:43:18,134	INFO tune.py:1033 -- Total run time: 102.07 seconds (102.05 seconds for the tuning loop).


24 trials in 105s


## 6.3 Reading the result

In [4]:
df = results.get_dataframe()
df = df[["MAE", "config/mechanism", "config/hidden", "config/n_experts", "config/lr"]]
df.columns = ["mae", "mechanism", "hidden", "n_experts", "lr"]

print(df.sort_values("mae").head(8).to_string(index=False))

print(f"\n{'mechanism':<10} {'trials':>7} {'best':>8} {'worst':>8}")
print("-" * 38)
for name, group in df.groupby("mechanism"):
    print(f"{name:<10} {len(group):>7} {group['mae'].min():>8.4f} {group['mae'].max():>8.4f}")

best_row = df.loc[df["mae"].idxmin()]
print(f"\nbaseline (plain, hidden=64, {SEEDS} seeds): {BASELINE:.4f} +/- {BASELINE_STD:.4f}")
print(f"best trial found by the search             : {best_row['mae']:.4f}  "
      f"({best_row['mechanism']}, hidden={int(best_row['hidden'])})")
print(f"beats baseline by more than 2 std           : "
      f"{best_row['mae'] < BASELINE - 2 * BASELINE_STD}")


     mae mechanism  hidden  n_experts       lr
0.262389     plain     128          4 0.000102
0.262406     plain     128          4 0.000103
0.262540     plain     128          2 0.000108
0.262925     plain     128          4 0.000120
0.263473      gate     128          2 0.000402
0.265875     plain     128          4 0.000251
0.265903     plain     128          4 0.000261
0.269358       moe     128          2 0.000363

mechanism   trials     best    worst
--------------------------------------
gate             5   0.2635   0.2917
highway          4   0.2763   0.2906
moe              6   0.2694   0.2935
plain            9   0.2624   0.3032

baseline (plain, hidden=64, 5 seeds): 0.2805 +/- 0.0021
best trial found by the search             : 0.2624  (plain, hidden=128)
beats baseline by more than 2 std           : True


Measured: `plain` wins, both the search's best trial (0.2624, hidden=128) and the
top of the per-mechanism table (`gate`'s best, 0.2635, is the closest challenger). Even
with a real daily cycle to route on, the router did not earn its parameters here
either -- a third measured case of the same finding, after `04_composition`'s
four-patch window and `05_gates_highways_moe`'s ETTh1 window. What the search *did*
find that mattered was width: hidden=128 beats the hidden=64 baseline by more than 2
standard deviations, entirely without any gate.

Read that carefully before generalising it. It is not "gating never helps" -- §1.4
warns against exactly that over-reach. It is: on three windows, from two datasets,
across a plain feedforward block, an ETTh1 sequence, and a pedestrian sequence with a
cycle a router could plausibly use, a plain block with the same or fewer parameters
matched or beat every gated alternative tried. That is a real, checkable pattern on
the data measured so far -- not a law, and not a reason to stop checking it on the
next dataset.

## 6.4 Log it, the way every search in this course does

One summary row for the whole search, not one per trial -- same reason
`1_pytorch_intro/04_search` 4.2 gives: a ray trial is its own process, and mlflow's
sqlite backend does not want concurrent writers.


In [5]:
import mlflow

MLRUNS = Path("../../artefacts/mlflow/mlruns.db").resolve()
mlflow.set_tracking_uri(f"sqlite:///{MLRUNS}")
mlflow.set_experiment("lesson2-search")
with mlflow.start_run(run_name="pedestrian_gate_search"):
    mlflow.log_params({"n_trials": N_TRIALS, "epochs": EPOCHS})
    mlflow.log_metric("baseline_mae", BASELINE)
    mlflow.log_metric("best_trial_mae", float(best_row["mae"]))
    mlflow.log_param("best_mechanism", best_row["mechanism"])
    mlflow.log_param("best_hidden", int(best_row["hidden"]))


2026/08/05 14:43:18 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/08/05 14:43:18 INFO mlflow.store.db.utils: Updating database tables


2026-08-05 14:43:18 INFO  [alembic.runtime.migration] Context impl SQLiteImpl.


2026-08-05 14:43:18 INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


2026-08-05 14:43:18 INFO  [alembic.runtime.migration] Context impl SQLiteImpl.


2026-08-05 14:43:18 INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


## What to take from this

- Same tool, same shape of search, second modality. If notebook 04 of
  `1_pytorch_intro` felt like a template rather than a one-off, that was the plan --
  D17 exists so students exercise the habit twice, not so this notebook discovers
  something new about ray.
- `05_gates_highways_moe` measured this exact component family losing on ETTh1's plain
  window. Whatever this search finds on `pedestrian_counts` is a second, independent
  data point on the same question, not a rerun of the first one -- read the two results
  side by side rather than picking whichever one you liked better.
- A search only teaches something if you named the baseline it has to beat, first,
  outside the search. §6.1 did that before §6.2 ran a single trial.

## Exercises

**E1 -- predict before you read §6.3.** Given `05_gates_highways_moe`'s result on
ETTh1 (Gate tied its capacity-matched control, Highway and MoE lost), write down what
you expect here, on a series with an actual daily cycle to route on. Which mechanism do
you expect to win, and why would a router have more to work with here than on ETTh1?

**E2 -- widen the space.** Add `w` to the search space (`tune.choice([96, 168, 336])`)
instead of holding it fixed. Does the best window interact with the best mechanism, or
are they independent? The parallel-categories style plot in
`1_pytorch_intro/04_search` 4.2 is the tool for exactly this question.

**E3 -- three seeds on the winner.** Re-run the best trial's config with 3 different
seeds (`torch.manual_seed`). Is the gap over the baseline still outside 2 standard
deviations, or did the search find a seed rather than a configuration?
